# Optical Cloud Removal: Minimal Reproducible Demo

This notebook provides a reproducible demonstration of optical cloud removal on Sentinel-2 satellite imagery using a compact U-Net baseline (`1,942,611` parameters).

### Key Highlights:
- **Reuses existing compact checkpoint**: Evaluates `checkpoints/compact/best_compact_unet.pt` without requiring retraining.
- **Correct display rendering**: Applies a shared 2%–98% percentile contrast stretch across Cloudy Input, U-Net Prediction, and Clear Reference.
- **Scientific evaluation**: Computes quantitative MAE and PSNR on raw physical surface reflectance $[0.0, 1.0]$.
- **Independent steps**: Inference and demo do not require full dataset re-downloading or GPU retraining.

## Step 1: Hardware & Environment Verification
Works seamlessly on both GPU (e.g. Google Colab T4) and CPU.

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Available VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("Running on CPU (sufficient for fast inference demo).")

## Step 2: Extract Project Source Archive and Install (`pip install -e .`)
Upload `project_source.zip` (exported from the local repository) or let the cell prompt for upload.

In [ ]:
import os, sys, zipfile
from pathlib import Path

target_dir = Path("/content/liss4-cloud-removal") if os.path.exists("/content") else Path(".").resolve()
target_dir.mkdir(parents=True, exist_ok=True)

zip_path = Path("project_source.zip")
if not zip_path.is_file() and not (target_dir / "src" / "cloud_removal").exists():
    try:
        from google.colab import files
        print("Please upload 'project_source.zip':")
        uploaded = files.upload()
    except Exception as e:
        print("Running in non-Colab environment or upload skipped:", e)

if zip_path.is_file():
    print(f"Unpacking {zip_path} into {target_dir}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(target_dir)

os.chdir(str(target_dir))
print(f"Current working directory: {os.getcwd()}")

# Editable install and required dependencies
!pip install -q -e .
!pip install -q rasterio mlflow pytest

# Ensure active Python kernel sees src/
src_dir = str(target_dir / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Verify imports in fresh execution
import cloud_removal
from cloud_removal.infer import load_model_from_checkpoint
from cloud_removal.demo import run_demo
print("SUCCESS: cloud_removal imported cleanly!")

## Step 3: Verify Checkpoint and Validation Data
Checks for `checkpoints/compact/best_compact_unet.pt` and `data/raw/sen12mscr/manifest.json`.
If missing, prompts to upload `unet_trained_gpu_artifacts.zip` or run optional dataset acquisition.

In [ ]:
import os, zipfile, shutil
from pathlib import Path

ckpt_file = Path("checkpoints/compact/best_compact_unet.pt")
if not ckpt_file.is_file():
    # Check for unet_trained_gpu_artifacts.zip
    art_zip = Path("unet_trained_gpu_artifacts.zip")
    if not art_zip.is_file() and not Path("checkpoints/best_compact_unet.pt").is_file():
        try:
            from google.colab import files
            print("Please upload 'unet_trained_gpu_artifacts.zip' (or best_compact_unet.pt):")
            files.upload()
        except Exception as e:
            print("Upload prompt:", e)
    if art_zip.is_file():
        print("Unpacking artifacts archive...")
        with zipfile.ZipFile(art_zip, "r") as z:
            z.extractall(".")

    # Resolve path if extracted directly into checkpoints/best_compact_unet.pt or root
    if Path("checkpoints/best_compact_unet.pt").is_file() and not ckpt_file.is_file():
        ckpt_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2("checkpoints/best_compact_unet.pt", ckpt_file)
    elif Path("best_compact_unet.pt").is_file() and not ckpt_file.is_file():
        ckpt_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2("best_compact_unet.pt", ckpt_file)

if ckpt_file.is_file():
    print(f"Checkpoint verified: {ckpt_file} ({ckpt_file.stat().st_size:,} bytes)")
else:
    print("WARNING: Checkpoint not found at checkpoints/compact/best_compact_unet.pt")

manifest_file = Path("data/raw/sen12mscr/manifest.json")
if manifest_file.is_file():
    print(f"Dataset manifest verified: {manifest_file}")
else:
    print("NOTE: Dataset manifest not found yet. Run Step 5 to acquire validation patches before running the demo.")


## Step 4: Run Minimal Demo Workflow
Executes end-to-end inference on a validation patch, computes physical reflectance metrics, and displays the result with shared 2%–98% contrast stretch.

In [ ]:
from cloud_removal.demo import run_demo
from IPython.display import Image, display
from pathlib import Path

ckpt_file = Path("checkpoints/compact/best_compact_unet.pt")
manifest_file = Path("data/raw/sen12mscr/manifest.json")

if ckpt_file.is_file() and manifest_file.is_file():
    demo_report = run_demo(
        checkpoint_path=ckpt_file,
        manifest_path=manifest_file,
        split="val",
        sample_index=0,
        output_figure="reports/figures/demo_comparison.png",
        device="auto"
    )
    
    fig_path = demo_report.get("comparison_figure")
    if fig_path and Path(fig_path).is_file():
        print(f"\nDisplaying Demo Comparison Figure ({fig_path}):")
        display(Image(filename=fig_path))
else:
    print("Prerequisites missing. Please ensure checkpoint and dataset manifest are present.")

## Step 5 (Optional): Acquire Official SEN12MS-CR Subset
Acquires the 719-pair Sentinel-2 optical subset (ROI 102, Winter) from official MediaTUM FTP if not already downloaded.

In [ ]:
from pathlib import Path
manifest_file = Path("data/raw/sen12mscr/manifest.json")

if not manifest_file.is_file():
    print("Acquiring 719-pair SEN12MS-CR subset from MediaTUM...")
    !python -m cloud_removal.acquire_sen12mscr --output-dir data/raw/sen12mscr
else:
    print(f"Dataset already acquired: {manifest_file}")

# Validate dataset integrity
!python -m cloud_removal.sen12mscr --manifest data/raw/sen12mscr/manifest.json

## Step 6 (Optional): Run Preflight Test Suite
Runs pytest unit tests covering U-Net architecture, dataset loader, metrics, and visualization.

In [ ]:
!pytest tests/test_unet.py tests/test_metrics.py tests/test_sen12mscr.py -v

## Step 7 (Optional): Full Split Evaluation & Figure Generation
Evaluates all 144 validation patches and regenerates side-by-side comparison figures for samples 0, 1, 2, 3.

In [ ]:
!python -m cloud_removal.infer \
    --checkpoint checkpoints/compact/best_compact_unet.pt \
    --eval-split val \
    --save-indices 0,1,2,3 \
    --output-dir reports/figures/compact

from IPython.display import Image, display
import glob
for fig in sorted(glob.glob("reports/figures/compact/inference_val_*.png")):
    print(f"Displaying {fig}:")
    display(Image(filename=fig))